# Notebook 01 — The Qubit and Superposition

**Prerequisites:** Go programming basics and a working gonb installation (see setup below).

Welcome to the first notebook in our 16-part quantum computing curriculum built on the **Goqu** SDK. By the end of this notebook you will be able to:

1. Describe what a qubit is mathematically (a 2D complex vector).
2. Build and visualize quantum circuits in Go.
3. Explain superposition, probability amplitudes, and the Born rule.
4. Demonstrate, with code, that superposition is **not** a coin flip.

---

### Prerequisites

You need the **gonb** Go kernel for Jupyter. Install it with:

```bash
go install github.com/janpfeifer/gonb@latest
gonb --install
```

Then open this notebook in JupyterLab or VS Code and select the **Go (gonb)** kernel.

In [ ]:
import (
	"fmt"
	"math"
	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/sim/statevector"
	"github.com/splch/goqu/viz"
)

## What is a qubit?

A classical bit is either 0 or 1. A **qubit** (quantum bit) is a two-dimensional complex vector:

$$|\psi\rangle = \alpha\,|0\rangle + \beta\,|1\rangle$$

where $\alpha$ and $\beta$ are **probability amplitudes** — complex numbers satisfying $|\alpha|^2 + |\beta|^2 = 1$.

The **computational basis** consists of two orthogonal states:

| State | Vector | Meaning |
|-------|--------|---------|
| $\|0\rangle$ | $\begin{pmatrix}1\\0\end{pmatrix}$ | "off", ground state |
| $\|1\rangle$ | $\begin{pmatrix}0\\1\end{pmatrix}$ | "on", excited state |

When we **measure** a qubit, we get outcome 0 with probability $|\alpha|^2$ and outcome 1 with probability $|\beta|^2$. This is the **Born rule**.

The Hadamard gate **H** transforms $|0\rangle$ into the **equal superposition** state:

$$H|0\rangle = \frac{1}{\sqrt{2}}|0\rangle + \frac{1}{\sqrt{2}}|1\rangle \;=\; |+\rangle$$

In this state, each outcome has probability $(1/\sqrt{2})^2 = 1/2$.

## Your first circuit

Let's build a single-qubit circuit that applies the Hadamard gate, then print its text diagram.

The Goqu builder uses a **fluent API**:
- `builder.New(name, nQubits)` creates a new circuit builder.
- `.H(0)` applies Hadamard to qubit 0.
- `.Build()` finalizes the circuit.
- `draw.String(circuit)` renders an ASCII diagram.

In [ ]:
%%
c, err := builder.New("hadamard", 1).H(0).Build()
if err != nil {
	fmt.Println("Error:", err)
} else {
	fmt.Println(draw.String(c))
}

## Inspecting the statevector

A **statevector simulator** tracks the exact amplitudes of every basis state. There is no randomness until we measure.

After applying H to $|0\rangle$, we expect the statevector to be:

$$\left[\frac{1}{\sqrt{2}},\; \frac{1}{\sqrt{2}}\right] \approx [0.707,\; 0.707]$$

Let's verify with `statevector.New(1)` (one qubit, initialized to $|0\rangle$).

In [ ]:
%%
sim := statevector.New(1)
c, _ := builder.New("h", 1).H(0).Build()
sim.Evolve(c)
sv := sim.StateVector()
fmt.Printf("Statevector: %v\n", sv)
fmt.Printf("|0> amplitude: %.6f   probability: %.4f\n", real(sv[0]), real(sv[0])*real(sv[0]))
fmt.Printf("|1> amplitude: %.6f   probability: %.4f\n", real(sv[1]), real(sv[1])*real(sv[1]))

## Visualizing on the Bloch sphere

Every single-qubit pure state can be represented as a point on the **Bloch sphere**:

- $|0\rangle$ is the **north pole**.
- $|1\rangle$ is the **south pole**.
- $|+\rangle$ sits on the **equator** (pointing along the +X axis).

The `viz.Bloch` function takes a 2-element statevector and returns an SVG rendering of the sphere.

In [ ]:
%%
sim := statevector.New(1)
c, _ := builder.New("bloch-plus", 1).H(0).Build()
sim.Evolve(c)
svg := viz.Bloch(sim.StateVector(), viz.WithTitle("|+> state on the Bloch sphere"))
gonbui.DisplayHTML(svg)

## Measurement and the Born rule

When we add `.MeasureAll()` to the circuit and run it for many **shots**, each shot collapses the superposition into a definite outcome. The fraction of shots yielding each outcome converges to the Born-rule probabilities.

For $|+\rangle$, we expect roughly 50% "0" and 50% "1".

In [ ]:
%%
c, _ := builder.New("measure-plus", 1).H(0).MeasureAll().Build()
sim := statevector.New(1)
counts, _ := sim.Run(c, 1000)
fmt.Println("Counts:", counts)
gonbui.DisplayHTML(viz.Histogram(counts, viz.WithTitle("H|0> — 1000 shots")))

## Predict, then verify

Before running the next cell, **pause and predict**: what will the measurement outcomes be for a circuit that applies only an **X gate** (the quantum NOT gate) to $|0\rangle$?

The X gate flips a qubit:

$$X|0\rangle = |1\rangle, \qquad X|1\rangle = |0\rangle$$

So we should **always** get outcome "1" — 100% of the time.

In [ ]:
%%
c, _ := builder.New("x-gate", 1).X(0).MeasureAll().Build()
fmt.Println(draw.String(c))
sim := statevector.New(1)
counts, _ := sim.Run(c, 1000)
fmt.Println("Counts:", counts)

## Superposition is NOT a coin flip

A common misconception is that a qubit in superposition is "just a coin flip — we don't know the answer yet, but it's already decided." This is wrong. A qubit in superposition carries **phase information** that can produce **interference**, something no classical coin can do.

### The proof: H followed by H

If superposition were merely classical ignorance (a coin flip), then:
- H would create a 50/50 coin.
- A second H would flip the coin again — still 50/50.

But quantum mechanics says:

$$H \cdot H |0\rangle = |0\rangle$$

The two Hadamard gates produce **constructive interference** for $|0\rangle$ and **destructive interference** for $|1\rangle$, perfectly canceling the $|1\rangle$ amplitude. The result is deterministically $|0\rangle$ — 100% of the time.

Mathematically:

$$H|+\rangle = H\left(\frac{|0\rangle + |1\rangle}{\sqrt{2}}\right) = \frac{H|0\rangle + H|1\rangle}{\sqrt{2}} = \frac{(|0\rangle + |1\rangle) + (|0\rangle - |1\rangle)}{2} = |0\rangle$$

Let's verify this and contrast it with what a "coin flip" model would predict.

## Self-Check Questions

Before attempting the exercises, make sure you can answer these:

1. True or false: A qubit in superposition is in state |0⟩ AND state |1⟩ at the same time. Why or why not?
2. Why does H·H|0⟩ = |0⟩, but a coin flipped twice still gives a 50/50 outcome?
3. If a qubit is in state |ψ⟩ = √(0.3)|0⟩ + √(0.7)|1⟩, what is the probability of measuring |0⟩?

In [ ]:
%%
// Quantum: H -> H should give |0> with 100% probability (interference!)
cHH, _ := builder.New("H-H interference", 1).H(0).H(0).MeasureAll().Build()
fmt.Println("Circuit: H -> H")
fmt.Println(draw.String(cHH))

sim := statevector.New(1)

// Check the statevector before measurement
cHHnoMeasure, _ := builder.New("H-H sv", 1).H(0).H(0).Build()
sim.Evolve(cHHnoMeasure)
fmt.Printf("Statevector after H->H: %v\n\n", sim.StateVector())

// Run with measurement
counts, _ := sim.Run(cHH, 1000)
fmt.Println("Quantum result (H->H, 1000 shots):")
fmt.Println(counts)
fmt.Println()
fmt.Println("If superposition were a coin flip, we'd expect ~500 '0' and ~500 '1'.")
fmt.Println("Instead we get 1000 '0' and 0 '1' — proof of quantum interference!")

---

## Exercises

Try these on your own before peeking at the hints.

### Exercise 1 — Deterministic |1>

Build a circuit that **always** measures `1`. Run it for 1000 shots and display the histogram.

*Hint: Which single gate flips $|0\rangle$ to $|1\rangle$?*

In [ ]:
%%
// Exercise 1: Build a circuit that always measures |1>
// Replace the ... with the correct gate call
// c, _ := builder.New("always-one", 1). ... .MeasureAll().Build()
// sim := statevector.New(1)
// counts, _ := sim.Run(c, 1000)
// gonbui.DisplayHTML(viz.Histogram(counts, viz.WithTitle("Exercise 1: Always |1>")))
fmt.Println("Uncomment the code above and fill in the gate!")

### Exercise 2 — A 75/25 split

Create a circuit that measures `0` with 75% probability and `1` with 25% probability. Run it for 1000 shots and display the histogram.

*Hint: The RY gate rotates the qubit on the Bloch sphere. If you want $P(|0\rangle) = \cos^2(\theta/2) = 0.75$, solve for $\theta$. The `math.Acos` and `math.Sqrt` functions will help.*

In [ ]:
%%
// Exercise 2: Build a circuit that gives 75% |0> and 25% |1>
// theta := 2 * math.Acos(math.Sqrt( ... ))
// c, _ := builder.New("75-25", 1).RY(theta, 0).MeasureAll().Build()
// sim := statevector.New(1)
// counts, _ := sim.Run(c, 1000)
// gonbui.DisplayHTML(viz.Histogram(counts, viz.WithTitle("Exercise 2: 75/25 split")))
_ = math.Sqrt // suppress unused import
fmt.Println("Uncomment the code above and fill in the probability!")

---

## Key takeaways

1. **A qubit** is a unit vector in a 2D complex vector space, written $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$.

2. **Superposition** means $\alpha$ and $\beta$ can both be non-zero simultaneously. The qubit is not "secretly" 0 or 1 — it is genuinely in both states at once.

3. **The Born rule** says measurement outcome probabilities are the squared magnitudes of the amplitudes: $P(0) = |\alpha|^2$, $P(1) = |\beta|^2$.

4. **Interference** is the hallmark of quantum mechanics. Amplitudes can add constructively or destructively, as we saw with $H \cdot H|0\rangle = |0\rangle$. No classical probability model can reproduce this.

5. **The Bloch sphere** is a geometric way to visualize single-qubit states, with $|0\rangle$ at the north pole, $|1\rangle$ at the south pole, and superposition states on the surface.

---

**Next up:** Notebook 02 — Single-Qubit Gates, where we'll explore all the ways to manipulate a single qubit.